# Akurasi label: section-context head

Buka dari root **SongMap**, lalu **Run All**. Notebook tetap menjalankan Python dari `.venv-songformer311` di sibling **Section-Annotations** secara eksplisit.

Eksperimen ini melatih **head tambahan 304.776 parameter** yang membaca ringkasan fitur audio semua section asli dalam satu lagu, posisi dan durasinya. Attention memungkinkan satu section memakai konteks section lainnya. Koreksinya ditambahkan ke log probabilitas `worship-v3` yang sudah tersimpan. Ini model tambahan; bobot SongFormer asli tetap disimpan.

Label manual **train** menjadi target training. Section yang ditahan tetap menyediakan konteks audio, tetapi tidak menyediakan label untuk loss. Validation yang sama dipakai untuk early stopping dan pemilihan epoch terbaik; test tidak dipakai dalam eksperimen ini. Boundary tetap sama.

Mulai dari koreksi nol, sehingga prediksi awal sama dengan `worship-v3`. Gunakan hasil kalibrasi sebelumnya sebagai pembanding tambahan, bukan sebagai target training. Peningkatan akurasi belum dijamin.


In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess, json

SONGMAP_ROOT=Path.cwd().resolve()
assert (SONGMAP_ROOT/'scripts/train_section_context.py').is_file(), 'Buka dari root SongMap'
SONGSTRUCT_ROOT=(SONGMAP_ROOT/'../Section-Annotations').resolve()
RUN_PYTHON=SONGSTRUCT_ROOT/'.venv-songformer311/bin/python'
assert RUN_PYTHON.is_file(), f'Python tidak ditemukan: {RUN_PYTHON}'

# Jika laporan kalibrasi disimpan di lokasi lain, ganti CALIBRATION dengan Path('/lokasi/comparison.json').
files=list((SONGMAP_ROOT/'outputs.local').glob('bridge-calibration-*/comparison.json'))
assert files, 'Hasil bridge_calibration_experiment.ipynb belum ditemukan'
CALIBRATION=max(files,key=lambda p:p.stat().st_mtime).resolve()
reference=json.loads(CALIBRATION.read_text())
assert reference.get('full_validation') and reference.get('split')=='val', 'Laporan validation belum lengkap'

# CPU dipakai hanya untuk head kecil ini; encoder tidak dijalankan. MPS juga didukung.
DEVICE='cpu'
cache=SONGMAP_ROOT/'outputs.local/section-context-cache-v1'
output=SONGMAP_ROOT/'outputs.local'/('section-context-'+datetime.now().strftime('%Y%m%d-%H%M%S'))
subprocess.run([str(RUN_PYTHON),'-c',
    "import sys,numpy,torch; print('Python:',sys.executable); print('NumPy:',numpy.__version__,'Torch:',torch.__version__)"],check=True)
print('Laporan kalibrasi:',CALIBRATION)
print('Cache section:',cache)
print('Run:',output)
print('Akurasi/F1 pembanding:',reference['baseline']['section_accuracy'],reference['baseline']['macro_f1_present'])
print('F1 kalibrasi sebelumnya:',reference['candidate']['macro_f1_present'])


In [ ]:
def run_logged(command,log):
    log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('a',encoding='utf-8') as handle:
        process=subprocess.Popen(command,cwd=SONGMAP_ROOT,stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:
        raise RuntimeError(f'Gagal ({status}); lihat {log}. Cache dan epoch yang sudah tersimpan tetap tersedia.')

def command():
    return [str(RUN_PYTHON),'-u',str(SONGMAP_ROOT/'scripts/train_section_context.py'),
        '--songstruct-root',str(SONGSTRUCT_ROOT),'--calibration',str(CALIBRATION),
        '--device',DEVICE,'--cache-dir',str(cache),'--output-dir',str(output)]

# Uji satu lagu train + satu lagu val, backward dan optimizer; tidak menyimpan checkpoint training.
run_logged(command()+['--smoke-test'],Path(str(output)+'.smoke.log'))


Cell berikut merangkum cache audio untuk train dan validation **sekali**, lalu melatih head dengan konteks satu lagu penuh. Cache section dapat digunakan ulang. Persiapan awal membaca file cache encoder yang besar; ini bukan inferensi encoder baru.

Training memakai maksimal 40 epoch, dropout 0,25, regularisasi koreksi, dan early stopping setelah 6 epoch tanpa kenaikan macro F1 validation. Head awal dengan koreksi nol ikut menjadi kandidat epoch 0. Tidak ada pencarian banyak konfigurasi otomatis.

Jika dihentikan, jalankan ulang **cell training berikut** tanpa mengubah variabel `output`. Script melanjutkan checkpoint epoch terakhir. Jika notebook/kernel ditutup, isi `output=Path('/lokasi/run/section-context-...')` setelah cell setup, baru jalankan cell training. Epoch yang terputus diulang dari checkpoint terakhir.

Coverage di bawah 98% pada section pendek tidak menghentikan proses jika seluruh kekurangannya cocok dengan ujung cache yang terpotong maksimal 120 ms per chunk. Kasus ini dicatat di `coverage_diagnostics.json`; fitur yang hilang tidak diisi dan label/boundary tidak diubah. Gap yang tidak cocok tetap menghasilkan error. Cache section yang sudah selesai tetap dipakai.


In [ ]:
if (output/'comparison.json').is_file():
    print('Training sudah selesai:',output/'comparison.json')
else:
    args=command()
    if (output/'run.json').is_file():
        args+=['--resume']
        print('Melanjutkan:',output)
    # Log berada di sebelah folder run, agar folder run tidak dibuat sebelum script memvalidasinya.
    run_logged(args,Path(str(output)+'.training.log'))


In [ ]:
report=json.loads((output/'comparison.json').read_text())
print('Best epoch:',report['best_epoch'],'| validation:',report['tracks'],'lagu')
for key in ('baseline','previous_calibration','candidate'):
    score=report[key]
    print(f"{key}: accuracy={score['section_accuracy']:.2%}; macro F1={score['macro_f1_present']:.4f}")
print('F1 per kelas (worship-v3 → context head):')
for label in report['baseline']['label_order']:
    before=report['baseline']['per_class'][label]
    after=report['candidate']['per_class'][label]
    print(f"  {label}: {before['f1']:.4f} → {after['f1']:.4f}; recall={after['recall']:.2%}; support={after['support']}")
changes=report['changed_sections']
print('Section berubah:',len(changes),
      '| salah → benar:',sum(not s['baseline_correct'] and s['candidate_correct'] for s in changes),
      '| benar → salah:',sum(s['baseline_correct'] and not s['candidate_correct'] for s in changes))
print('F1 delta vs kalibrasi sebelumnya:',report['macro_f1_delta_from_previous_calibration'])
if report['macro_f1_delta_from_previous_calibration']<=0:
    print('Head konteks belum mengungguli kalibrasi sebelumnya; jangan mengganti pipeline aplikasi.')
print('Kirim file:',output/'comparison.json')
print('Checkpoint head tambahan:',output/'best.pt')

coverage=report.get("audio_coverage")
if coverage:
    print("Section dengan tail cache yang tercatat:",coverage["reviewed_sections_below_98_percent"])
    print("Detail coverage:",coverage["details_file"])


Bagikan **comparison.json**. Model tambahan ini tidak kompatibel sebagai pengganti langsung `worship-v3/best.pt` dan belum dipasang ke aplikasi. Pemeriksaan seluruh kelas, termasuk chorus, tetap diperlukan sebelum memilih kandidat berikutnya.

Validation telah dipakai untuk beberapa eksperimen, sehingga kenaikan pada set ini belum menjadi bukti final generalisasi. Boundary/outro yang terlambat adalah masalah terpisah; training ini mempelajari label pada batas section yang tetap.
